# AdamW lawan NAdam pada CNN-LSTM

> **Tentang berkas ini.** Notebook ini bagian dari repositori penelitian
> deteksi deepfake audio di <https://github.com/Tristan-tech-ai/general-AI>. Seluruh kode yang
> dijalankan di sini diambil langsung dari repositori itu, tidak ada yang
> ditempel ke dalam notebook, sehingga hasilnya sebanding dengan hasil yang
> dilaporkan di sana. Laporan lengkapnya ada pada `NASKAH.pdf`.

Notebook ini menjalankan **model yang sama persis dua kali**. Data sama, split
sama, augmentasi sama, jumlah putaran sama, batch sama, inisialisasi acak sama.
Satu satunya yang berbeda adalah **optimizer**-nya.

Itu memang syarat perbandingan yang sah. Kalau dua hal berubah sekaligus,
selisih hasilnya tidak bisa dibebankan pada salah satunya.

| langkah | isi |
|---|---|
| 1 | Menyiapkan runtime, kode, dan dependensi |
| 2 | Mengambil dan memverifikasi dataset |
| 3 | Apa itu optimizer, dan apa bedanya kedua ini |
| 4 | Melatih dengan **AdamW** |
| 5 | Melatih dengan **NAdam** |
| 6 | Membandingkan keduanya |

**Sebelum mulai:** menu `Runtime` lalu `Change runtime type`, pilih **T4 GPU**,
lalu `Save`.

> **Waktu.** Ini yang paling ringan dari empat model proposal, karena tidak ada encoder besar yang harus dilewati. Sepuluh putaran biasanya selesai dalam hitungan menit. Notebook ini melatih **dua kali**, jadi
> kalikan dua. Untuk memperlihatkan prosesnya saja, turunkan `EPOCHS` menjadi
> 3. Yang penting kedua sisi memakai angka yang sama, dan sel di langkah 5
> memaksa itu.

Untuk penjelasan model dan datanya sendiri, lihat `Proses_CNN_LSTM.ipynb`.

## Langkah 1 · Menyiapkan runtime, kode, dan dependensi

Yang terjadi di sel ini, berurutan:

1. **Memeriksa kartu grafisnya.** Bukan sekadar ada atau tidak, tetapi jenisnya,
   karena kartu lama dan kartu baru memakai format bilangan yang berbeda saat
   melatih. Ini berpengaruh pada hasil, jadi dicatat sejak awal.
2. **Mengambil kode penelitian** dari GitHub. Tidak ada kode yang ditempel di
   dalam notebook ini, semuanya berasal dari repositori yang sama dengan yang
   dipakai di komputer lokal. Itu syarat supaya angkanya bisa dibandingkan.
3. **Memasang pustaka** yang belum dibawa Colab.
4. **Menjalankan pengaman konfigurasi**, yaitu skrip yang menolak melanjutkan
   bila ada pengaturan yang menyimpang dari acuan penelitian.

In [ ]:
# Notebook ini khusus satu model. Ketiga nilai berikut sengaja ditulis tetap,
# bukan sebagai pilihan, supaya tidak ada yang tergeser tanpa sengaja dan
# angkanya tetap sebanding dengan hasil rujukan di mesin lokal.
MODEL      = "cnnlstm"
BATCH      = 32          # batch yang dipakai di mesin lokal untuk model ini
AUGMENTASI = "codec"    # perbaikan kebocoran codec, lihat notebook Proses

import os, subprocess, sys

AKAR = "/content/general-ai"
if not os.path.exists(os.path.join(AKAR, ".git")):
    subprocess.run(["git", "clone", "--quiet", "https://github.com/Tristan-tech-ai/general-AI.git", AKAR], check=True)
os.chdir(AKAR)
sys.path.insert(0, AKAR)

from colab.siapkan import siapkan
siapkan(MODEL)

## Langkah 2 · Mengambil dan memverifikasi dataset

Dataset yang dipakai adalah **Fake-or-Real, potongan dua detik**: 17.870 berkas
suara, masing masing tepat dua detik, 16.000 sampel per detik, satu kanal.

Yang penting di sini bukan mengunduhnya, tapi **membuktikan datanya sama**.
Kalau dataset di Colab berbeda sedikit saja dari yang dipakai di komputer
lokal, seluruh perbandingan angka menjadi tidak berarti.

Karena itu ada dua lapis pemeriksaan:

1. **Sidik jari arsip.** Kode sha256 dari berkas arsipnya dibandingkan dengan
   yang tercatat di repositori.
2. **Sidik jari pohon berkas.** Ini yang lebih dalam: setiap berkas wav
   diperiksa satu per satu, lalu diringkas menjadi satu kode. Pemeriksaan ini
   tidak peduli berkasnya datang dari mana atau namanya di folder apa. Yang
   dibandingkan isinya.

Kalau salah satu tidak cocok, sel ini **berhenti** dan tidak melanjutkan ke
pelatihan. Itu memang disengaja.

In [ ]:
from colab.siapkan import siapkan_dataset, verifikasi_dan_manifest

siapkan_dataset(buat_cache=True)
baris = verifikasi_dan_manifest()

## Langkah 3 · Apa itu optimizer, dan apa bedanya kedua ini

**Optimizer adalah aturan yang menentukan seberapa jauh dan ke arah mana bobot
model digeser setiap kali ia salah.** Modelnya sendiri tidak berubah. Yang
berubah hanya cara menuruni bukitnya.

Analogi yang biasanya langsung dimengerti: bayangkan menuruni lembah berkabut,
hanya bisa merasakan kemiringan tanah di bawah kaki.

- **Adam** melangkah mengikuti kemiringan, sambil mengingat arah beberapa
  langkah terakhir supaya tidak zigzag. Ingatan arah itu namanya momentum.
- **NAdam** melakukan hal yang sama, tapi menambahkan satu hal: sebelum
  melangkah, ia **melihat dulu ke arah yang sedang dituju momentumnya**, lalu
  mengukur kemiringan **di titik itu**, bukan di tempat ia berdiri sekarang.
  Ini yang disebut momentum Nesterov.

Efeknya: bila lembahnya menikung, NAdam sadar lebih cepat dan tidak terlanjur
melewati tikungannya.

Sel di bawah memperlihatkannya pada permukaan mainan dua dimensi yang bentuknya
menyerupai lembah sempit. Keduanya diberi titik awal, laju, dan jumlah langkah
yang persis sama.

> **Jujur soal batasnya.** Permukaan mainan ini **bukan bukti** bahwa NAdam
> lebih baik untuk tugas kita. Ia hanya memperlihatkan apa yang berbeda secara
> mekanis. Buktinya baru datang dari langkah 4 sampai 6, dan itu pun perlu
> beberapa inisialisasi sebelum layak disebut kesimpulan.

**Satu hal yang perlu diluruskan.** Pembandingnya di sini bukan Adam polos,
melainkan **AdamW**, yaitu Adam dengan peluruhan bobot terpisah. Itu yang
dipakai di seluruh hasil penelitian ini sejak awal. Supaya perbandingannya
adil, NAdam juga dijalankan dengan peluruhan bobot terpisah, sehingga
**satu satunya yang berbeda memang suku Nesterov-nya**, bukan dua hal sekaligus.

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt

def permukaan(x, y):
    # lembah sempit melengkung: sedikit salah arah langsung terasa
    return (1 - x) ** 2 + 20 * (y - x ** 2) ** 2

def jalur(Opt, **kw):
    p = torch.tensor([-1.5, 2.0], requires_grad=True)
    o = Opt([p], lr=0.05, **kw)
    t = [p.detach().clone().numpy()]
    for _ in range(120):
        o.zero_grad()
        permukaan(p[0], p[1]).backward()
        o.step()
        t.append(p.detach().clone().numpy())
    return np.array(t)

t_adam = jalur(torch.optim.AdamW, weight_decay=0.0)
t_nadam = jalur(torch.optim.NAdam, weight_decay=0.0)

gx, gy = np.meshgrid(np.linspace(-2, 2, 300), np.linspace(-0.5, 3, 300))
gz = permukaan(gx, gy)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2), layout="constrained")
ax[0].contour(gx, gy, np.log10(gz + 1e-6), levels=25, cmap="Greys", linewidths=0.6)
ax[0].plot(*t_adam.T, color="#1864AB", lw=1.8, label="AdamW")
ax[0].plot(*t_nadam.T, color="#E8590C", lw=1.8, label="NAdam")
ax[0].scatter([-1.5], [2.0], c="k", s=30, zorder=5)
ax[0].scatter([1.0], [1.0], marker="*", c="#2F9E44", s=200, zorder=5,
              label="titik terendah")
ax[0].set_title("Jalur turun, 120 langkah, laju sama")
ax[0].legend(); ax[0].set_xlabel("bobot 1"); ax[0].set_ylabel("bobot 2")

for t, nama_o, warna in [(t_adam, "AdamW", "#1864AB"), (t_nadam, "NAdam", "#E8590C")]:
    ax[1].plot([permukaan(*q) for q in t], color=warna, lw=1.8, label=nama_o)
ax[1].set_yscale("log"); ax[1].set_title("Seberapa salah, tiap langkah")
ax[1].set_xlabel("langkah"); ax[1].set_ylabel("nilai fungsi (skala log)")
ax[1].legend(); ax[1].grid(alpha=0.25)
plt.show()

print(f"setelah 120 langkah  AdamW: {permukaan(*t_adam[-1]):.4f}")
print(f"                     NAdam: {permukaan(*t_nadam[-1]):.4f}")
print()
print("Sekali lagi: ini permukaan mainan. Ia menunjukkan APA yang berbeda,")
print("bukan MANA yang lebih baik untuk deteksi deepfake audio.")

## Langkah 4 · Melatih dengan AdamW

Ini sisi pembanding, yaitu pengaturan yang dipakai di seluruh hasil penelitian
ini sejak awal. Angka yang keluar dari sini harus dekat dengan angka lokal
untuk model yang sama, dan bila jauh melenceng itu pertanda ada yang salah
sebelum kita membandingkan apa pun.

Setelan di sel ini berlaku untuk **kedua** sisi. Langkah 5 mengambil angka yang
sama dari sini, sehingga tidak mungkin tanpa sengaja membandingkan sepuluh
putaran melawan tiga putaran.

In [ ]:
#@title Setelan pelatihan, berlaku untuk KEDUA sisi { display-mode: "form" }
EPOCHS = 10 #@param {type:"slider", min:1, max:20, step:1}
SEED = 42 #@param {type:"integer"}
SIMPAN_HASIL_KE_DRIVE = False #@param {type:"boolean"}

import time
from colab.siapkan import jalankan, berhenti

# Tag dibentuk dengan aturan yang sama seperti train.py, jadi langkah 6 tahu
# persis dua folder mana yang harus dibandingkan tanpa menebak.
TAG_A = f"{MODEL}_official_{AUGMENTASI}_b{BATCH}e{EPOCHS}_s{SEED}"
TAG_B = f"{MODEL}_official_{AUGMENTASI}NAD_b{BATCH}e{EPOCHS}_s{SEED}"
NAMA_A, NAMA_B = "AdamW", "NAdam"

def latih(tambahan, tag, nama_sisi):
    print(f"\n{'=' * 68}\n  {nama_sisi}\n{'=' * 68}")
    print(f"epoch {EPOCHS}   batch {BATCH}   seed {SEED}")
    print(f"keluaran runs_colab/{tag}\n")
    t0 = time.time()
    kode, _ = jalankan([
        sys.executable, "train.py",
        "--model", MODEL, "--split", "official", "--augment", AUGMENTASI,
        "--epochs", str(EPOCHS), "--batch", str(BATCH), "--workers", "2",
        "--seed", str(SEED), "--out", "runs_colab", *tambahan,
    ])
    if kode != 0:
        berhenti("Pelatihan gagal. Kalau pesannya menyebut kehabisan memori "
                 "GPU, turunkan BATCH menjadi 8 lalu jalankan lagi.")
    print(f"\nselesai dalam {(time.time() - t0) / 60:.1f} menit")

latih([], TAG_A, f"SISI A: {NAMA_A}")

## Langkah 5 · Melatih dengan NAdam

Sekarang sisi yang diuji. Perintahnya **sama persis** dengan langkah 4 kecuali
satu bendera tambahan: `--optimizer nadam`.

Seed-nya juga sama, yaitu nilai yang diisi pada langkah 4. Ini penting dan
sering terlewat: bila seed-nya berbeda, selisih yang muncul dapat berasal dari
inisialisasi acaknya, bukan dari optimizer-nya. Dengan seed yang sama, kedua
model berangkat dari titik awal yang identik.

In [ ]:
latih(['--optimizer', 'nadam'], TAG_B, f"SISI B: {NAMA_B}")

if SIMPAN_HASIL_KE_DRIVE:
    from colab.siapkan import simpan_ke_drive
    simpan_ke_drive()

## Langkah 6 · Membandingkan keduanya

Tiga hal yang dibaca, dan urutannya penting.

**Kurva belajar.** Grafik pertama menunjukkan EER validasi di tiap putaran.
Ini memperlihatkan **bagaimana** keduanya belajar, bukan hanya di mana mereka
berakhir. Dua model bisa berakhir di angka yang sama lewat jalan yang sangat
berbeda, dan yang lebih cepat stabil punya nilainya sendiri.

**Angka akhir pada data uji.** Tabelnya memuat akurasi pada dua ambang, AUC,
dan EER. AUC dan EER tidak bergantung ambang, jadi keduanya pembanding yang
lebih jujur daripada akurasi.

**Ukuran selisihnya.** Ini yang paling penting dan paling sering dilewatkan.
Selisih beberapa poin **belum tentu berarti apa apa**, karena menjalankan model
yang sama dengan inisialisasi acak berbeda pun sudah menghasilkan selisih
sebesar itu. Sel di bawah membandingkan selisih yang terukur dengan ragam antar
inisialisasi yang sudah diukur di penelitian ini, lalu menyatakan terus terang
apakah selisihnya layak disebut nyata atau belum.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from forlib.metrics import full_metrics, prior_matched_threshold

RAGAM_LOKAL = {"wav2vec2": 0.51, "ast": 2.94, "hubert": 2.16, "cnnlstm": 2.28}

def muat(tag):
    d = f"runs_colab/{tag}"
    if not os.path.exists(f"{d}/results.json"):
        raise SystemExit(f"Belum ada hasil di {d}. Jalankan langkah 4 dan 5 dulu.")
    y, p, _ = np.load(f"{d}/test_scores.npy")
    return json.load(open(f"{d}/results.json")), y.astype(int), p

res_a, y, p_a = muat(TAG_A)
res_b, _, p_b = muat(TAG_B)

# ---- kurva belajar
fig, ax = plt.subplots(1, 2, figsize=(13, 3.6), layout="constrained")
for res, nama_s, warna in [(res_a, NAMA_A, "#1864AB"), (res_b, NAMA_B, "#E8590C")]:
    h = res["history"]
    ax[0].plot([e["epoch"] for e in h], [e["val_eer"] * 100 for e in h],
               marker="o", ms=3.5, color=warna, lw=1.8, label=nama_s)
    ax[1].plot([e["epoch"] for e in h], [e["loss"] for e in h],
               marker="o", ms=3.5, color=warna, lw=1.8, label=nama_s)
ax[0].set_title("EER validasi tiap putaran (makin rendah makin baik)")
ax[0].set_xlabel("putaran"); ax[0].set_ylabel("EER validasi (persen)")
ax[1].set_title("Loss latih tiap putaran")
ax[1].set_xlabel("putaran"); ax[1].set_ylabel("loss")
for a in ax:
    a.legend(); a.grid(alpha=0.25)
plt.show()

# ---- angka akhir
def angka(p):
    return (full_metrics(y, p, 0.5), full_metrics(y, p, prior_matched_threshold(p, 0.5)))

(a05, apm), (b05, bpm) = angka(p_a), angka(p_b)
print(f"{'ukuran':<28}{NAMA_A:>18}{NAMA_B:>18}{'selisih':>11}")
print("-" * 76)
baris_tabel = [
    ("akurasi @ ambang 0,5", a05["accuracy"] * 100, b05["accuracy"] * 100, "%"),
    ("akurasi @ disesuaikan", apm["accuracy"] * 100, bpm["accuracy"] * 100, "%"),
    ("AUC", a05["auc"], b05["auc"], ""),
    ("EER (makin kecil baik)", a05["eer"] * 100, b05["eer"] * 100, "%"),
    ("putaran terbaik", res_a["best_epoch"], res_b["best_epoch"], "int"),
]
for nama_u, va, vb, sat in baris_tabel:
    if sat == "%":
        print(f"{nama_u:<28}{va:>17.2f}%{vb:>17.2f}%{vb - va:>+10.2f}")
    elif sat == "int":
        print(f"{nama_u:<28}{va:>18d}{vb:>18d}{vb - va:>+11d}")
    else:
        print(f"{nama_u:<28}{va:>18.4f}{vb:>18.4f}{vb - va:>+11.4f}")
menit_a = sum(e["sec"] for e in res_a["history"]) / 60
menit_b = sum(e["sec"] for e in res_b["history"]) / 60
print(f"{'waktu latih (menit)':<28}{menit_a:>18.1f}{menit_b:>18.1f}{menit_b - menit_a:>+11.1f}")
print("-" * 76)

# ---- laju dropout yang dipelajari, kalau ada
if "dropout_dipelajari" in res_b:
    laju = res_b["dropout_dipelajari"]
    print(f"\nlaju dropout yang DIPELAJARI model: "
          f"{', '.join(f'{v:.4f}' for v in laju)}")
    print("laju tetap yang dipakai sisi A    : 0,2000")
    arah = "lebih banyak" if laju[0] > 0.2 else "lebih sedikit"
    print(f"Model memilih menjatuhkan {arah} unit daripada tebakan manusia.")

# ---- apakah selisihnya layak disebut nyata
selisih = (bpm["accuracy"] - apm["accuracy"]) * 100
ragam = RAGAM_LOKAL[MODEL]
print(f"\nselisih akurasi yang terukur          : {selisih:+.2f} poin")
print(f"ragam antar inisialisasi model ini    : sekitar {ragam:.2f} poin")
if abs(selisih) < ragam:
    print("\nSelisihnya LEBIH KECIL daripada ragam antar inisialisasi.")
    print("Dengan satu inisialisasi, ini BELUM bisa disebut perbedaan.")
    print("Ulangi langkah 4 dan 5 dengan SEED 1337 lalu 2024 sebelum menyimpulkan.")
else:
    print("\nSelisihnya LEBIH BESAR daripada ragam antar inisialisasi satu model.")
    print("Itu menjanjikan, tetapi satu inisialisasi tetap belum cukup.")
    print("Ulangi dengan SEED 1337 lalu 2024 untuk memastikan.")

## Penutup

Satu inisialisasi tidak cukup untuk menyimpulkan optimizer mana yang lebih
baik. Itu bukan kehati hatian berlebihan, melainkan temuan yang diukur langsung
di penelitian ini: menjalankan HuBERT delapan kali dengan pengaturan yang persis
sama menghasilkan akurasi antara 93,93 dan 99,45 persen. Jaraknya 5,52 poin,
dari model yang sama.

Untuk CNN-LSTM, ragam antar inisialisasi pada konfigurasi ini sekitar
**simpangan 2,28 atas tiga inisialisasi**.

Cara menutup perbandingan ini dengan benar:

1. Jalankan langkah 4 sampai 6 dengan `SEED = 42`, lalu `1337`, lalu `2024`.
2. Ambil rata rata tiap sisi beserta simpangannya.
3. Baru bandingkan rata ratanya, dan hanya sebut berbeda bila selisihnya
   melampaui simpangan itu.

Sebelum ketiganya selesai, yang didukung data dari notebook ini hanyalah
selisih yang teramati pada satu inisialisasi. Pernyataan bahwa salah satu
optimizer lebih baik adalah klaim yang berbeda dan belum berdasar. Batas itu
disengaja, karena penelitian ini justru menemukan bahwa sebagian kesimpulan
yang ditarik terlalu dini akhirnya tidak bertahan.

Dua belas notebook di rangkaian ini:

| model | proses | optimizer | dropout |
|---|---|---|---|
| Wav2Vec2 | `Proses_Wav2Vec2.ipynb` | `Optimizer_Wav2Vec2.ipynb` | `Dropout_Wav2Vec2.ipynb` |
| AST (Audio Spectrogram Transformer) | `Proses_AST.ipynb` | `Optimizer_AST.ipynb` | `Dropout_AST.ipynb` |
| HuBERT | `Proses_HuBERT.ipynb` | `Optimizer_HuBERT.ipynb` | `Dropout_HuBERT.ipynb` |
| CNN-LSTM | `Proses_CNN_LSTM.ipynb` | `Optimizer_CNN_LSTM.ipynb` | `Dropout_CNN_LSTM.ipynb` |